In [52]:
from langchain_community.llms import Ollama

model = Ollama(model="phi3")

In [64]:
from langchain_core.tools import tool

@tool
def greet(name:str) -> str:
    "Greet user."
    return (f"Hello {name}! You look great")


@tool 
def evenOdd(x: int) -> str:
    "Check if number is odd"
    if (x % 2 == 0):
        return "Even"
    else:
        return "Odd"


@tool 
def squared(x: int) -> int:
    "Square a number"
    return x * x

@tool 
def user (name: str, age:int) ->str:
    "User info"
    return (f"The user name is {name}. He/She is {age} years old")

tools=[greet,evenOdd,squared,user]

In [65]:
greet.invoke("Luis")

'Hello Luis! You look great'

In [66]:
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

greet(name: str) -> str - Greet user.
evenOdd(x: int) -> str - Check if number is odd
squared(x: int) -> int - Square a number
user(name: str, age: int) -> str - User info


In [82]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage

system_prompt = f"""\
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.

Respond ONLY with the JSON blob.

Example:
User input: Greeting to Ahmed
Output: {{"name": "greet", "arguments": {{"name": "Ahmed"}}}}

Example:
User input: Is 7 odd or even?
Output: {{"name": "evenOdd", "arguments": {{"x": 7}}}}

Example:
User input: What is 9 squared?
Output: {{"name": "squared", "arguments": {{"x": 9}}}}

Example:
User input: Sara 31
Output: {{"name": "user", "arguments": {{"name": "Sara", "age": 31}}}}
"""

prompt = ChatPromptTemplate.from_messages(
    [SystemMessage(content=system_prompt), ("user", "{input}")]
)

In [91]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]

    if requested_tool is None:
        available = ", ".join(tool_name_to_tool)
        return f"Error: tool '{name}' does not exist. Available tools: {available}"

    try:
        return requested_tool.invoke(tool_call_request["arguments"], config=config)
    except Exception as e:
        return f"Error while running tool '{name}': {e}"
    

In [84]:
chain =  prompt | model | JsonOutputParser() | invoke_tool

In [85]:
chain.invoke({"input": "Greeting to Salma"})

'Hello Salma! You look great'

In [86]:
chain.invoke({"input": "Is the number 122 even or odd"})

'Even'

In [88]:
chain.invoke({"input": "what's the square of 4"})

16

In [93]:
chain.invoke({"input": "What is the capital of France?"})

KeyError: 'None'

The model can not answer a question that has no tool related to. 
    
To fix that, we may add an example to the system promt and then handle it in the invoke tool. 
    Example:
    User input: What is the capital of France?
    Output: {{"name": "none", "arguments": {{"reason": "No tool can answer geography questions."}}}}
    